# Data Engineer (итерация 1)

# Data Engineer Report — Fake Job Postings

**Источник:** `data/raw/fake_job_postings.csv`  
**Выход:** `/Users/iuriipostnii/Desktop/ГП3/gp3/data/processed/cleaned.csv`  
**Target:** `fraudulent` (бинарный, сильно несбалансированный ~5% класс 1)

## Бизнес-контекст
Бинарная классификация мошеннических вакансий для HR-площадки. Цель — снизить ручную модерацию и защитить пользователей от скам-постингов. Ключевая метрика — F1 / recall класса 1 при контроле precision.

## План очистки
1. Загрузить CSV, посмотреть shape и долю NaN.
2. Спрофилировать колонки: dtypes, % NaN, уникальность, распределение target.
3. Разделить колонки на группы: `target`, `text` (as_is), `numeric`, `categorical_low` (<20 уникальных → OHE), `categorical_high` (>50 уникальных → frequency encoding), `drop` (>70% NaN или бесполезные id).
4. Применить стратегии: impute (mean/median/mode), clip outliers (1–99 перцентиль), one-hot, frequency encoding.
5. Текстовые колонки (`title`, `company_profile`, `description`, `requirements`, `benefits`) оставить as_is — их фичеризацией займётся DS (TF-IDF / embeddings).
6. Сохранить `cleaned.csv`.

## Запреты (чего НЕ делаем)
- Не трогаем `fraudulent` (target).
- Не делаем target encoding — это утечка, работа DS под CV.
- Не балансируем классы — работа DS.
- Не заполняем 0 в salary-полях (0 имеет смысл и искажает распределение).
- Не делаем OHE на колонках с тысячами категорий.

In [ ]:
import pandas as pd
import numpy as np

DF = pd.read_csv("data/raw/fake_job_postings.csv")
print("Shape:", DF.shape)
print("\nNaN per column:")
print(DF.isna().sum())
print("\nTotal NaN:", DF.isna().sum().sum())

Shape: (17880, 18)

NaN per column:
job_id                     0
title                      0
location                 346
department             11547
salary_range           15012
company_profile         3308
description                1
requirements            2696
benefits                7212
telecommuting              0
has_company_logo           0
has_questions              0
employment_type         3471
required_experience     7050
required_education      8105
industry                4903
function                6455
fraudulent                 0
dtype: int64

Total NaN: 70106


## Профиль данных

Смотрим типы, долю пропусков, уникальность и распределение target. На основе этого раскладываем колонки по группам для разных стратегий очистки.

In [ ]:
TARGET = "fraudulent"

# Профиль: dtype, NaN-доля, уникальность
profile = pd.DataFrame({
    "dtype": DF.dtypes.astype(str),
    "nan_frac": DF.isna().mean().round(3),
    "n_unique": DF.nunique(dropna=True),
})
print("=== Column profile ===")
print(profile)

print("\n=== Target distribution ===")
print(DF[TARGET].value_counts())
print("Positive rate:", round(DF[TARGET].mean(), 4))

# Раскладка колонок (по знанию датасета fake_job_postings)
TEXT_COLS = [c for c in ["title", "company_profile", "description", "requirements", "benefits"] if c in DF.columns]
ID_COLS  = [c for c in ["job_id"] if c in DF.columns]

# Всё, что не target / text / id — кандидаты на очистку
candidate_cols = [c for c in DF.columns if c not in TEXT_COLS + ID_COLS + [TARGET]]

NUM_COLS, CAT_LOW, CAT_HIGH, DROP_HIGH_NAN = [], [], [], []
for c in candidate_cols:
    nan_frac = DF[c].isna().mean()
    if nan_frac > 0.70:
        DROP_HIGH_NAN.append(c)
        continue
    if pd.api.types.is_numeric_dtype(DF[c]):
        NUM_COLS.append(c)
    else:
        nun = DF[c].nunique(dropna=True)
        if nun < 20:
            CAT_LOW.append(c)
        elif nun > 50:
            CAT_HIGH.append(c)
        else:
            # 20..50 — кладём в low (OHE) если не слишком много, иначе frequency
            if nun <= 30:
                CAT_LOW.append(c)
            else:
                CAT_HIGH.append(c)

print("\nTEXT_COLS:", TEXT_COLS)
print("ID_COLS:", ID_COLS)
print("NUM_COLS:", NUM_COLS)
print("CAT_LOW (OHE):", CAT_LOW)
print("CAT_HIGH (freq enc):", CAT_HIGH)
print("DROP (>70% NaN):", DROP_HIGH_NAN)

=== Column profile ===
                     dtype  nan_frac  n_unique
job_id               int64     0.000     17880
title                  str     0.000     11231
location               str     0.019      3105
department             str     0.646      1337
salary_range           str     0.840       874
company_profile        str     0.185      1709
description            str     0.000     14801
requirements           str     0.151     11967
benefits               str     0.403      6204
telecommuting        int64     0.000         2
has_company_logo     int64     0.000         2
has_questions        int64     0.000         2
employment_type        str     0.194         5
required_experience    str     0.394         7
required_education     str     0.453        13
industry               str     0.274       131
function               str     0.361        37
fraudulent           int64     0.000         2

=== Target distribution ===
fraudulent
0    17014
1      866
Name: count, dtype: in

## Стратегия и применение

| Группа | Стратегия |
|---|---|
| `fraudulent` (target) | не трогаем |
| `job_id` | drop (идентификатор) |
| text (title, description, ...) | as_is — фичеризация на стороне DS |
| числовые | impute (mean если ~симметрично, иначе median) + clip по 1/99 перцентилям. Для `salary_*` 0 не используем как заполнитель. |
| категориальные <20 уник. | mode impute → one-hot |
| категориальные >50 уник. (напр. `location`, `department`, `salary_range`, `industry`) | mode impute → frequency encoding (count / total) |
| NaN > 70% | drop column |

In [ ]:
actions = []  # журнал действий для отчёта

# 1) Drop job_id (идентификатор, не фича)
for c in ID_COLS:
    DF = DF.drop(columns=[c])
    actions.append((c, "drop", "identifier, not a feature"))

# 2) Drop колонок с >70% NaN
for c in DROP_HIGH_NAN:
    DF = DF.drop(columns=[c])
    actions.append((c, "drop_column", ">70% NaN — слишком мало сигнала"))

# 3) Числовые: impute + clip (1–99 перцентиль)
for c in NUM_COLS:
    if c not in DF.columns:
        continue
    s = DF[c]
    # Выбор mean/median по skew. NaN — тут заполнителем не 0.
    skew = s.skew()
    if pd.notna(skew) and abs(skew) < 0.5:
        fill_val = s.mean()
        strat = "impute_mean"
        reason = f"numeric, skew={skew:.2f} ~симметрично"
    else:
        fill_val = s.median()
        strat = "impute_median"
        reason = f"numeric, skew={skew:.2f} — устойчиво к выбросам"
    DF[c] = DF[c].fillna(fill_val)
    actions.append((c, strat, reason))

    # Clip по 1/99 перцентилям — только если есть реальный разброс
    lo, hi = DF[c].quantile(0.01), DF[c].quantile(0.99)
    if pd.notna(lo) and pd.notna(hi) and hi > lo:
        DF[c] = DF[c].clip(lo, hi)
        actions.append((c, "clip_1_99", f"обрезка выбросов в [{lo:.3f}, {hi:.3f}]"))

# 4) Категориальные low-cardinality: mode impute + OHE
for c in list(CAT_LOW):
    if c not in DF.columns:
        continue
    mode_val = DF[c].mode(dropna=True)
    mode_val = mode_val.iloc[0] if len(mode_val) else "missing"
    DF[c] = DF[c].fillna(mode_val).astype(str)
    actions.append((c, "impute_mode", f"categorical, mode='{mode_val}'"))

    dummies = pd.get_dummies(DF[c], prefix=c, drop_first=False, dtype=int)
    DF = pd.concat([DF.drop(columns=[c]), dummies], axis=1)
    actions.append((c, "one_hot", f"<=30 уникальных, {dummies.shape[1]} новых колонок"))

# 5) Категориальные high-cardinality: mode impute + frequency encoding
for c in list(CAT_HIGH):
    if c not in DF.columns:
        continue
    mode_val = DF[c].mode(dropna=True)
    mode_val = mode_val.iloc[0] if len(mode_val) else "missing"
    DF[c] = DF[c].fillna(mode_val).astype(str)
    actions.append((c, "impute_mode", f"categorical high-card, mode='{mode_val}'"))

    freq = DF[c].value_counts(normalize=True)
    DF[c + "_freq"] = DF[c].map(freq).astype(float)
    DF = DF.drop(columns=[c])
    actions.append((c, "frequency_encoding", "count/total — избегаем взрыва размерности и утечки таргета"))

# 6) Текстовые — оставляем as_is (DS решит: TF-IDF, embeddings, длины и т.п.)
for c in TEXT_COLS:
    if c in DF.columns:
        # только заполним пропуски пустой строкой, чтобы downstream не падал
        n_nan = DF[c].isna().sum()
        DF[c] = DF[c].fillna("")
        actions.append((c, "as_is (fillna='')", f"текст — фичеризация на стороне DS; заполнено {n_nan} NaN пустой строкой"))

# 7) Target — не трогаем, только сверяем
assert TARGET in DF.columns, "Target потерян!"
actions.append((TARGET, "keep", "target не трогаем"))

# Сохраняем
import os
out_path = "/Users/iuriipostnii/Desktop/ГП3/gp3/data/processed/cleaned.csv"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
DF.to_csv(out_path, index=False)

print("Saved to:", out_path)
print("Final shape:", DF.shape)
print("Remaining NaN total:", DF.isna().sum().sum())
print("Target still present:", TARGET in DF.columns)
print("Target distribution:")
print(DF[TARGET].value_counts())

Saved to: /Users/iuriipostnii/Desktop/ГП3/gp3/data/processed/cleaned.csv
Final shape: (17880, 38)
Remaining NaN total: 0
Target still present: True
Target distribution:
fraudulent
0    17014
1      866
Name: count, dtype: int64


## Применённые действия

In [ ]:
actions_df = pd.DataFrame(actions, columns=["column", "strategy", "reason"])
print(actions_df.to_string(index=False))

             column           strategy                                                               reason
             job_id               drop                                            identifier, not a feature
       salary_range        drop_column                                      >70% NaN — слишком мало сигнала
      telecommuting      impute_median                            numeric, skew=4.51 — устойчиво к выбросам
      telecommuting          clip_1_99                                    обрезка выбросов в [0.000, 1.000]
   has_company_logo      impute_median                           numeric, skew=-1.46 — устойчиво к выбросам
   has_company_logo          clip_1_99                                    обрезка выбросов в [0.000, 1.000]
      has_questions        impute_mean                                      numeric, skew=0.03 ~симметрично
      has_questions          clip_1_99                                    обрезка выбросов в [0.000, 1.000]
    employment_type        i